# NB05 — Frontend Web Vitals

Uses **Playwright** (headless Chromium) to measure real browser performance metrics
for each QCanvas page:

| Metric | What it measures |
|--------|------------------|
| **LCP** | Largest Contentful Paint — when main content renders |
| **FCP** | First Contentful Paint — first pixel appears |
| **CLS** | Cumulative Layout Shift — visual stability |
| **TTFB** | Time to First Byte — server responsiveness |
| **Load** | Full page load time |
| **Bundle sizes** | JS/CSS download weights from Resource Timing API |

**Requires:** Frontend running at `http://localhost:3000` AND `playwright install chromium`  
**Output:** `../results/frontend/vitals.csv`, `../results/frontend/bundle_sizes.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

try:
    from playwright.sync_api import sync_playwright
    PLAYWRIGHT_OK = True
    print('✓ Playwright available')
except ImportError:
    PLAYWRIGHT_OK = False
    print('✗ Playwright not installed.')
    print('  Run: pip install playwright && playwright install chromium')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

from playwright_helpers import (
    benchmark_pages, get_js_bundle_sizes, collect_page_vitals,
    FRONTEND_URL,
) if PLAYWRIGHT_OK else (None, None, None, None)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})

RESULTS_DIR = Path('../results/frontend')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PAGES = [
    {'label': 'Landing Page', 'path': '/'},
    {'label': 'Login',        'path': '/login'},
    {'label': 'Signup',       'path': '/signup'},
    {'label': 'IDE (app)',    'path': '/app'},
    {'label': 'Examples',     'path': '/examples'},
    {'label': 'About',        'path': '/about'},
]

N_RUNS = 3   # runs per page for averaging
print(f'Will test {len(PAGES)} pages × {N_RUNS} runs = {len(PAGES)*N_RUNS} measurements')

## 1 — Collect Web Vitals for All Pages

In [ ]:
if not PLAYWRIGHT_OK:
    raise RuntimeError('Playwright is required. Install it and re-run this cell.')

print(f'Measuring Web Vitals at {FRONTEND_URL} ...')
vitals_raw = benchmark_pages(PAGES, headless=True, n_runs=N_RUNS, frontend_url=FRONTEND_URL)

df_vitals = pd.DataFrame(vitals_raw)
df_vitals.to_csv(RESULTS_DIR / 'vitals_raw.csv', index=False)
print(f'\n✓ Saved vitals_raw.csv ({len(df_vitals)} rows)')
display(df_vitals[['label', 'run', 'fcp_ms', 'lcp_ms', 'cls', 'ttfb_ms', 'load_ms']].round(1))

## 2 — Aggregate and Summarise

In [ ]:
numeric_cols = ['fcp_ms', 'lcp_ms', 'cls', 'ttfb_ms', 'load_ms', 'nav_latency_ms']
# Only average numeric cols that exist
existing_cols = [c for c in numeric_cols if c in df_vitals.columns]

df_agg = df_vitals.groupby('label')[existing_cols].mean().reset_index()
df_agg.to_csv(RESULTS_DIR / 'vitals.csv', index=False)
print('✓ Saved vitals.csv')

# Good/Needs Improvement/Poor thresholds (Core Web Vitals 2024)
def lcp_rating(ms):
    if ms is None or pd.isna(ms): return '?'
    return '✓ Good' if ms < 2500 else ('~ Needs Impr.' if ms < 4000 else '✗ Poor')

def fcp_rating(ms):
    if ms is None or pd.isna(ms): return '?'
    return '✓ Good' if ms < 1800 else ('~ Needs Impr.' if ms < 3000 else '✗ Poor')

def cls_rating(val):
    if val is None or pd.isna(val): return '?'
    return '✓ Good' if val < 0.1 else ('~ Needs Impr.' if val < 0.25 else '✗ Poor')

if 'lcp_ms' in df_agg: df_agg['LCP Rating'] = df_agg['lcp_ms'].map(lcp_rating)
if 'fcp_ms' in df_agg: df_agg['FCP Rating'] = df_agg['fcp_ms'].map(fcp_rating)
if 'cls' in df_agg: df_agg['CLS Rating'] = df_agg['cls'].map(cls_rating)

display(df_agg.round(2))

## 3 — Web Vitals Bar Charts

In [ ]:
def color_by_lcp(ms):
    return 'seagreen' if ms < 2500 else ('goldenrod' if ms < 4000 else 'tomato')

pages = df_agg['label'].tolist()
has_lcp = 'lcp_ms' in df_agg.columns
has_fcp = 'fcp_ms' in df_agg.columns
has_cls = 'cls' in df_agg.columns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if has_lcp:
    colors = [color_by_lcp(v) for v in df_agg['lcp_ms'].fillna(9999)]
    axes[0].barh(pages, df_agg['lcp_ms'].fillna(0), color=colors, alpha=0.85)
    axes[0].axvline(2500, color='green', linestyle='--', linewidth=1, label='Good <2.5s')
    axes[0].axvline(4000, color='orange', linestyle='--', linewidth=1, label='Poor >4s')
    axes[0].set_xlabel('LCP (ms)')
    axes[0].set_title('Largest Contentful Paint')
    axes[0].legend(fontsize=8)

if has_fcp:
    axes[1].barh(pages, df_agg['fcp_ms'].fillna(0), color='steelblue', alpha=0.85)
    axes[1].axvline(1800, color='green', linestyle='--', linewidth=1, label='Good <1.8s')
    axes[1].set_xlabel('FCP (ms)')
    axes[1].set_title('First Contentful Paint')
    axes[1].legend(fontsize=8)

if has_cls:
    cls_colors = ['seagreen' if v < 0.1 else ('goldenrod' if v < 0.25 else 'tomato')
                  for v in df_agg['cls'].fillna(0)]
    axes[2].barh(pages, df_agg['cls'].fillna(0), color=cls_colors, alpha=0.85)
    axes[2].axvline(0.1, color='green', linestyle='--', linewidth=1, label='Good <0.1')
    axes[2].set_xlabel('CLS Score')
    axes[2].set_title('Cumulative Layout Shift')
    axes[2].legend(fontsize=8)

plt.suptitle('QCanvas Frontend — Core Web Vitals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'core_web_vitals.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved core_web_vitals.png')

## 4 — Load Time Timeline (Waterfall Overview)

In [ ]:
if 'ttfb_ms' in df_agg.columns and 'load_ms' in df_agg.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    y = range(len(pages))
    ax.barh(y, df_agg['load_ms'].fillna(0), color='lightsteelblue', label='Full Load')
    ax.barh(y, df_agg.get('fcp_ms', pd.Series([0]*len(df_agg))).fillna(0), color='steelblue', label='FCP')
    ax.barh(y, df_agg['ttfb_ms'].fillna(0), color='navy', label='TTFB')
    ax.set_yticks(list(y))
    ax.set_yticklabels(pages)
    ax.set_xlabel('Time (ms)')
    ax.set_title('Page Load Breakdown (TTFB → FCP → Full Load)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'load_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved load_waterfall.png')

## 5 — JS Bundle Sizes

In [ ]:
print('Collecting bundle sizes from IDE page (most JS-heavy)...')
with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    ctx = browser.new_context(viewport={'width': 1440, 'height': 900})
    page = ctx.new_page()
    entries = get_js_bundle_sizes(page, url=f'{FRONTEND_URL}/app')
    ctx.close()
    browser.close()

df_bundles = pd.DataFrame(entries)
if not df_bundles.empty:
    df_bundles = df_bundles.sort_values('transfer_size_kb', ascending=False)
    df_bundles.to_csv(RESULTS_DIR / 'bundle_sizes.csv', index=False)

    total_js_kb = df_bundles[df_bundles['type'] == 'script']['transfer_size_kb'].sum()
    total_css_kb = df_bundles[df_bundles['type'] == 'link']['transfer_size_kb'].sum()
    print(f'Total JS transferred: {total_js_kb:.1f} KB ({total_js_kb/1024:.2f} MB)')
    print(f'Total CSS transferred: {total_css_kb:.1f} KB')

    fig, ax = plt.subplots(figsize=(12, 5))
    top10 = df_bundles.head(10)
    short_names = [n.split('/')[-1][:40] for n in top10['name']]
    colors = ['steelblue' if t == 'script' else 'orchid' for t in top10['type']]
    ax.barh(short_names, top10['transfer_size_kb'], color=colors, alpha=0.85)
    ax.set_xlabel('Transfer Size (KB)')
    ax.set_title('Top 10 Largest Resources (IDE Page)')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='steelblue', label='JS'), Patch(facecolor='orchid', label='CSS')])
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'bundle_sizes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved bundle_sizes.csv + bundle_sizes.png')
else:
    print('⚠ No resource entries found — page may not have loaded correctly')